In [74]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.svm import SVR
from sklearn.metrics import root_mean_squared_error, r2_score


In [50]:
df=pd.read_csv(r"diabetes_012_health_indicators_BRFSS2015.csv")

In [51]:
df= df.iloc[0:1000, :]

In [52]:
df.shape[0]

1000

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Diabetes_012          1000 non-null   float64
 1   HighBP                1000 non-null   float64
 2   HighChol              1000 non-null   float64
 3   CholCheck             1000 non-null   float64
 4   BMI                   1000 non-null   float64
 5   Smoker                1000 non-null   float64
 6   Stroke                1000 non-null   float64
 7   HeartDiseaseorAttack  1000 non-null   float64
 8   PhysActivity          1000 non-null   float64
 9   Fruits                1000 non-null   float64
 10  Veggies               1000 non-null   float64
 11  HvyAlcoholConsump     1000 non-null   float64
 12  AnyHealthcare         1000 non-null   float64
 13  NoDocbcCost           1000 non-null   float64
 14  GenHlth               1000 non-null   float64
 15  MentHlth              

In [54]:
df.describe()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,...,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.463000,0.591000,0.530000,0.983000,29.217000,0.446000,0.066000,0.134000,0.636000,0.578000,...,0.95800,0.103000,2.883000,4.123000,5.766000,0.305000,0.348000,9.069000,4.769000,5.315000
std,0.831458,0.491895,0.499349,0.129336,6.218085,0.497324,0.248406,0.340823,0.481389,0.494126,...,0.20069,0.304111,1.099327,8.392515,9.929141,0.460638,0.476574,2.666712,1.052971,2.231312
min,0.000000,0.000000,0.000000,0.000000,16.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.00000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,8.000000,4.000000,3.000000
50%,0.000000,1.000000,1.000000,1.000000,28.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,1.00000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,9.000000,5.000000,6.000000
75%,0.000000,1.000000,1.000000,1.000000,32.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,1.00000,0.000000,4.000000,3.000000,7.000000,1.000000,1.000000,11.000000,6.000000,7.000000
max,2.000000,1.000000,1.000000,1.000000,59.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.00000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,8.000000


In [55]:
df.corr(numeric_only=True)

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
Diabetes_012,1.000000,0.240753,0.119608,0.073267,0.192749,0.023002,0.181464,0.151743,-0.116212,-0.033170,...,0.002675,0.028943,0.248783,0.005315,0.098618,0.213751,-0.002839,0.172481,-0.082375,-0.146675
HighBP,0.240753,1.000000,0.235429,0.110879,0.168136,0.005786,0.122833,0.195878,-0.079795,-0.018936,...,0.018475,0.000850,0.333474,0.073060,0.152339,0.268357,-0.007122,0.319148,-0.163265,-0.197146
HighChol,0.119608,0.235429,1.000000,0.093151,0.037716,0.087146,0.088930,0.164570,-0.091946,-0.005436,...,0.022574,0.022478,0.220661,0.095497,0.081165,0.145133,0.036006,0.191259,-0.058198,-0.096084
CholCheck,0.073267,0.110879,0.093151,1.000000,0.059358,-0.006505,0.034958,0.051730,0.061288,0.028601,...,0.126724,-0.108136,0.014158,-0.051559,0.014048,0.036712,0.047356,0.096277,0.029937,0.018574
BMI,0.192749,0.168136,0.037716,0.059358,1.000000,-0.045571,0.015345,0.014606,-0.082269,-0.046727,...,-0.042422,0.103567,0.205801,0.072186,0.066616,0.207525,-0.003552,-0.135704,-0.037896,-0.058897
Smoker,0.023002,0.005786,0.087146,-0.006505,-0.045571,1.000000,0.036981,0.095884,-0.069642,-0.096898,...,0.017371,0.040122,0.073570,0.083974,0.026224,0.043564,0.252527,0.026588,-0.053473,0.004068
Stroke,0.181464,0.122833,0.088930,0.034958,0.015345,0.036981,1.000000,0.191019,-0.075138,0.006948,...,-0.024657,0.108682,0.156602,0.010987,0.108135,0.200068,-0.008185,0.097385,-0.083252,-0.135069
HeartDiseaseorAttack,0.151743,0.195878,0.164570,0.051730,0.014606,0.095884,0.191019,1.000000,-0.074580,-0.002687,...,0.053094,0.059858,0.204857,0.070522,0.176401,0.198484,0.088547,0.202380,-0.072649,-0.093732
PhysActivity,-0.116212,-0.079795,-0.091946,0.061288,-0.082269,-0.069642,-0.075138,-0.074580,1.000000,0.077398,...,0.017739,-0.058175,-0.303756,-0.138063,-0.249251,-0.266246,0.029111,-0.038118,0.100549,0.103126
Fruits,-0.033170,-0.018936,-0.005436,0.028601,-0.046727,-0.096898,0.006948,-0.002687,0.077398,1.000000,...,0.053257,-0.023541,-0.061500,-0.071955,-0.068501,-0.049651,-0.098380,0.066940,0.066409,0.003568


In [56]:
df.columns

Index(['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='object')

In [57]:
X=df.drop("Diabetes_012", axis=1)
y=df["Diabetes_012"]

In [58]:
X_train, X_test, y_train, y_test =train_test_split(X, y, random_state=42, test_size=0.2)

In [59]:
model = SVR()

In [60]:
param_grid={"kernel":["linear", "poly", "rbf", "sigmoid"]}

In [61]:
grid_search = GridSearchCV(model, param_grid, scoring="neg_root_mean_squared_error")

In [62]:
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVR()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'kernel': ['linear', 'poly', ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and cand

In [63]:
grid_search.cv_results_

{'mean_fit_time': array([0.25159163, 0.02516079, 0.03306236, 0.02871351]),
 'std_fit_time': array([0.03140456, 0.00313947, 0.00924181, 0.00623692]),
 'mean_score_time': array([0.00495429, 0.00527816, 0.01031003, 0.0183074 ]),
 'std_score_time': array([0.00141593, 0.00130066, 0.00117726, 0.01965313]),
 'param_kernel': masked_array(data=['linear', 'poly', 'rbf', 'sigmoid'],
              mask=[False, False, False, False],
        fill_value=np.str_('?'),
             dtype=object),
 'params': [{'kernel': 'linear'},
  {'kernel': 'poly'},
  {'kernel': 'rbf'},
  {'kernel': 'sigmoid'}],
 'split0_test_score': array([-0.90406767, -0.8109108 , -0.82002366, -2.37566092]),
 'split1_test_score': array([-0.86593683, -0.85558345, -0.86115833, -2.19795284]),
 'split2_test_score': array([-0.87589504, -0.85617422, -0.86764771, -3.28644843]),
 'split3_test_score': array([-0.97844165, -0.9744454 , -0.97624115, -2.50556115]),
 'split4_test_score': array([-0.89582763, -0.88842961, -0.89367603, -2.83143793]

In [78]:
model = SVR(kernel="rbf")

In [79]:
model.fit(X_train, y_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [80]:
y_pred = model.predict(X_test)

In [81]:
print("RMSE", root_mean_squared_error(y_test, y_pred))

RMSE 0.9659515778668715


In [82]:
print("R2_Square", r2_score(y_test, y_pred))

R2_Square -0.22872421502353735


In [83]:
import joblib

In [84]:
joblib.dump(model, "svr_model.pkl")

['svr_model.pkl']

In [86]:
model=joblib.load("svr_model.pkl")

In [87]:
model.predict(X.sample(1).to_numpy())

c:\Users\rohit\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but SVR was fitted with feature names
  warnings.warn(


array([0.09018323])